# Trimestre: serial acquirer scorecard

Starter notebook. Run cells top to bottom (Shift+Enter).

**Before this notebook:** run `python pull.py ...` then `python explore.py flatten` in the terminal, so `data/tables/` exists.

**How it's built:** the Onesheet field names are unknown until kickoff, so the notebook *guesses* which tables and columns to use and prints its guess. If a guess is wrong, change the setting in the cell marked **SET ME** and rerun from there.

| Step | What you get |
| --- | --- |
| 1. Setup | Libraries loaded, folder checked |
| 2. Your universe | The 5 Canadian tickers + the US comparison set |
| 3. Explore | Which tables exist and what's in them |
| 4. Deals | Acquisitions per company per year, from dated events |
| 5. Metrics | Revenue, FCF, acquisition spend, ROIC per company per period |
| 6. Scorecard | One row per company, ranked, Canada vs US |
| 7. Charts | The two visuals for the demo |
| 8. Sources | The filing quote behind each number |
| 9. Export | Excel file for slides or sharing |

## 1. Setup

In [1]:
import os, re, glob, pathlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

TABLES = pathlib.Path("data/tables/combined")
print("Working folder:", os.getcwd())
print("Tables found:", TABLES.exists(), "-", len(list(TABLES.glob("*.csv"))) if TABLES.exists() else 0, "files")

Working folder: c:\Users\eglan\OneDrive - McGill University\Desktop\VsCode_Sheets\trimestre-toolkit
Tables found: False - 0 files


## 2. Your universe — SET ME

Use the exact ticker spelling that `pull.py` saved (the folder names in `data/raw/`).
Put the US index tickers in `US` if you pulled them too; leave it empty otherwise.

In [ ]:
CANADA = []   # e.g. ["CSU.TO", "TOI.V", "LMN.V", "...", "..."]
US     = []   # the organizers' US index, if pulled

# If left empty, use every symbol found in the data
all_symbols = sorted({p.name for p in pathlib.Path("data/raw").iterdir() if p.is_dir() and not p.name.startswith("_")})
if not CANADA and not US:
    CANADA = all_symbols
    print("CANADA / US not set; using every symbol found:", CANADA)

GROUP = {**{s: "Canada" for s in CANADA}, **{s: "US" for s in US}}
UNIVERSE = list(GROUP)
print(f"{len(CANADA)} Canadian, {len(US)} US")

## 3. Explore the tables

Biggest tables first. The deep profile ones start with `company_profile__` (or `pack__` if you used the data pack).

In [ ]:
index = pd.read_csv("data/tables/INDEX.csv")
index.head(25)

In [ ]:
def load(name):
    """Load a combined table by (part of) its name, keeping only your universe."""
    matches = [p for p in TABLES.glob("*.csv") if name in p.stem]
    if not matches:
        raise FileNotFoundError(f"No table containing '{name}'. Check INDEX.csv.")
    path = min(matches, key=lambda p: len(p.stem))   # shortest = most exact match
    df = pd.read_csv(path)
    if "symbol" in df and UNIVERSE:
        df = df[df["symbol"].isin(UNIVERSE)]
    print(f"{path.stem}: {len(df)} rows")
    return df

def pick(df, *keywords, required=True):
    """First column whose name contains any keyword (case-insensitive)."""
    for kw in keywords:
        for c in df.columns:
            if kw.lower() in c.lower():
                return c
    if required:
        raise KeyError(f"No column like {keywords}. Columns: {list(df.columns)}")
    return None

def search_tables(word):
    """Which tables have a column or a value containing this word?"""
    out = []
    for p in TABLES.glob("*.csv"):
        df = pd.read_csv(p, dtype=str)
        cols = [c for c in df.columns if word.lower() in c.lower()]
        vals = df.apply(lambda s: s.str.contains(word, case=False, na=False)).sum().sum()
        if cols or vals:
            out.append({"table": p.stem, "matching_columns": ", ".join(cols), "cells_mentioning": int(vals)})
    return pd.DataFrame(out).sort_values("cells_mentioning", ascending=False) if out else pd.DataFrame()

search_tables("acqui")

## 4. Deals: acquisitions from the dated events — SET ME

The guess: the table with "events" in its name, a date column, and a type/category column.

In [ ]:
EVENTS_TABLE = "events"      # part of the table name
EVENT_DATE   = None          # None = guess
EVENT_TYPE   = None          # None = guess
DEAL_WORDS   = r"acqui|merger|purchase of|bought"   # what counts as a deal

events = load(EVENTS_TABLE)
EVENT_DATE = EVENT_DATE or pick(events, "date")
EVENT_TYPE = EVENT_TYPE or pick(events, "type", "category", "kind", "title")
EVENT_TEXT = pick(events, "title", "description", "summary", "name", required=False)
print("date column:", EVENT_DATE, "| type column:", EVENT_TYPE, "| text column:", EVENT_TEXT)

events["_date"] = pd.to_datetime(events[EVENT_DATE], errors="coerce")
print("\nEvent types:")
print(events[EVENT_TYPE].value_counts().head(15))

In [ ]:
text = events[EVENT_TYPE].astype(str)
if EVENT_TEXT:
    text = text + " " + events[EVENT_TEXT].astype(str)
deals = events[text.str.contains(DEAL_WORDS, case=False, regex=True)].copy()
deals["year"] = deals["_date"].dt.year

deals_per_year = deals.pivot_table(index="symbol", columns="year", values=EVENT_DATE,
                                   aggfunc="count", fill_value=0)
deals_per_year["total"] = deals_per_year.sum(axis=1)
deals_per_year.sort_values("total", ascending=False)

## 5. Metrics — SET ME

The guess: the biggest "long" table with a metric name, a period and a value (one row per metric per period).
Then each scorecard input is matched to metric names by keywords. **Check the printed matches**; finance names vary a lot.

In [ ]:
METRICS_TABLE = None    # None = guess; or part of a table name from INDEX.csv

def guess_metrics_table():
    for t in index["table"]:
        df = pd.read_csv(TABLES / f"{t}.csv", nrows=200)
        has_val = any(re.search(r"value|amount", c, re.I) for c in df.columns)
        has_per = any(re.search(r"period|date|year|quarter", c, re.I) for c in df.columns)
        has_name = any(re.search(r"_key|metric|name|label", c, re.I) for c in df.columns)
        if has_val and has_per and has_name:
            return t
    return None

METRICS_TABLE = METRICS_TABLE or guess_metrics_table()
print("Metrics table:", METRICS_TABLE)
m = load(METRICS_TABLE)
M_NAME   = pick(m, "_key", "metric", "label", "name")
M_PERIOD = pick(m, "period", "fiscal", "date", "year")
M_VALUE  = pick(m, "value", "amount")
M_SOURCE = pick(m, "quote", "source", required=False)
print("name:", M_NAME, "| period:", M_PERIOD, "| value:", M_VALUE, "| source:", M_SOURCE)
m[M_VALUE] = pd.to_numeric(m[M_VALUE], errors="coerce")
print(f"\n{m[M_NAME].nunique()} distinct metrics. First 40:")
print(sorted(m[M_NAME].dropna().unique())[:40])

In [ ]:
# Scorecard inputs -> keywords to find them among the metric names. Edit freely.
INPUTS = {
    "revenue":     ["revenue", "sales"],
    "fcf":         ["free cash flow", "free_cash_flow", "fcf"],
    "acq_spend":   ["acquisition spend", "acquisitions, net", "acquisition_spend",
                    "business acquisitions", "cash paid for acquisitions"],
    "acq_count":   ["acquisitions_count", "number of acquisitions", "acquisitions completed"],
    "roic":        ["roic", "return on invested capital", "return_on_invested"],
    "net_debt":    ["net debt", "net_debt"],
}

names = pd.Series(m[M_NAME].dropna().unique())
MATCH = {}
for k, kws in INPUTS.items():
    hit = names[names.str.lower().apply(lambda n: any(w in n for w in kws))]
    MATCH[k] = list(hit)
    print(f"{k:10} -> {list(hit)[:5] if len(hit) else 'NOT FOUND (edit keywords or skip)'}")

In [ ]:
# Wide table: one row per company x period, one column per input.
# Periods are kept as reported; sort order assumes they sort chronologically (e.g. 2024Q4 < 2025Q1).
frames = []
for k, metric_names in MATCH.items():
    if not metric_names:
        continue
    sub = m[m[M_NAME].isin(metric_names[:1])]          # first match only; change if wrong
    frames.append(sub[["symbol", M_PERIOD, M_VALUE]].rename(columns={M_VALUE: k}))

wide = frames[0]
for f in frames[1:]:
    wide = wide.merge(f, on=["symbol", M_PERIOD], how="outer")
wide = wide.sort_values(["symbol", M_PERIOD])
wide

## 6. Scorecard

Each component is ranked as a percentile within your universe (0–100), then weighted.
These are **starting assumptions**, not a tested model; judge them against what you know about serial acquirers.

| Component | Calculation | Why |
| --- | --- | --- |
| Deal cadence | Deals per year (events) | Is the M&A engine running? |
| Reinvestment | Acquisition spend ÷ FCF, capped at 1 | Compounders redeploy cash; above 1 means borrowing to buy |
| Growth | Revenue growth, first to last period | Is the buying turning into scale? |
| Returns | Latest ROIC | Are deals still earning their cost? |
| Returns trend | ROIC change, first to last | Do returns hold as the company gets bigger? |

In [ ]:
WEIGHTS = {"deal_cadence": 0.20, "reinvestment": 0.20, "growth": 0.20, "roic": 0.25, "roic_trend": 0.15}

def first_last(s):
    s = s.dropna()
    return (s.iloc[0], s.iloc[-1]) if len(s) >= 2 else (np.nan, np.nan)

rows = []
years = max(deals["year"].nunique(), 1) if len(deals) else 1
for sym in UNIVERSE:
    w = wide[wide["symbol"] == sym]
    r = {"symbol": sym, "group": GROUP.get(sym, "?")}
    r["deal_cadence"] = (deals["symbol"] == sym).sum() / years
    if {"acq_spend", "fcf"} <= set(w):
        spend, fcf = w["acq_spend"].abs().sum(), w["fcf"].sum()
        r["acq_spend_to_fcf"] = spend / fcf if fcf else np.nan
        r["reinvestment"] = min(r["acq_spend_to_fcf"], 1) if pd.notna(r["acq_spend_to_fcf"]) else np.nan
    if "revenue" in w:
        a, b = first_last(w["revenue"])
        r["growth"] = b / a - 1 if a else np.nan
    if "roic" in w:
        a, b = first_last(w["roic"])
        r["roic"], r["roic_trend"] = b, b - a
    rows.append(r)

card = pd.DataFrame(rows).set_index("symbol")
used = {k: v for k, v in WEIGHTS.items() if k in card and card[k].notna().any()}
missing = set(WEIGHTS) - set(used)
if missing:
    print("Not enough data for:", missing, "-> weights rescaled over the rest")
total_w = sum(used.values())
card["score"] = sum(card[k].rank(pct=True) * 100 * w / total_w for k, w in used.items())
card = card.sort_values("score", ascending=False)
card.round(2)

In [ ]:
# Canada vs US (only meaningful if you pulled the US set)
card.groupby("group")[[c for c in card.columns if c not in ("group",)]].median(numeric_only=True).round(2)

## 7. Charts

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3 + 0.3 * len(card)))
colors = ["#2b6cb0" if g == "Canada" else "#a0aec0" for g in card["group"]]
ax.barh(card.index[::-1], card["score"][::-1], color=colors[::-1])
ax.set_xlabel("Score (0–100)")
ax.set_title("Serial acquirer scorecard")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# The core trade-off: how fast they buy vs whether returns hold up
if {"deal_cadence", "roic"} <= set(card) and card["roic"].notna().any():
    fig, ax = plt.subplots(figsize=(7, 5))
    for g, col in (("Canada", "#2b6cb0"), ("US", "#a0aec0")):
        d = card[card["group"] == g]
        ax.scatter(d["deal_cadence"], d["roic"], s=80, color=col, label=g)
        for sym, row in d.iterrows():
            ax.annotate(sym, (row["deal_cadence"], row["roic"]), xytext=(5, 3), textcoords="offset points", fontsize=9)
    ax.set_xlabel("Deals per year"); ax.set_ylabel("Latest ROIC")
    ax.set_title("Deal pace vs returns")
    ax.legend(frameon=False)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    plt.tight_layout(); plt.show()
else:
    print("Need deal cadence and ROIC for this chart.")

## 8. Sources: the quote behind each number

Use this in Q&A, or put the quotes in your slides.

In [ ]:
def sources(symbol, metric_key=None):
    """Filing quotes for a company, optionally for one scorecard input."""
    if not M_SOURCE:
        return "No source column found in the metrics table."
    sub = m[m["symbol"] == symbol]
    if metric_key:
        sub = sub[sub[M_NAME].isin(MATCH.get(metric_key, [])[:1])]
    return sub[[M_NAME, M_PERIOD, M_VALUE, M_SOURCE]]

top = card.index[0]
print("Top-ranked:", top)
sources(top, "revenue")

In [ ]:
# The deals behind the cadence, with their sources
deal_cols = [c for c in ["symbol", EVENT_DATE, EVENT_TYPE, EVENT_TEXT] if c] + \
            [c for c in deals.columns if re.search("quote|source|filing|url", c, re.I)]
deals[deals["symbol"] == top][deal_cols].sort_values(EVENT_DATE)

## 9. Export

In [ ]:
out = pathlib.Path("output"); out.mkdir(exist_ok=True)
with pd.ExcelWriter(out / "scorecard.xlsx") as xl:
    card.round(3).to_excel(xl, sheet_name="scorecard")
    deals_per_year.to_excel(xl, sheet_name="deals_per_year")
    wide.to_excel(xl, sheet_name="metrics", index=False)
    deals[deal_cols].to_excel(xl, sheet_name="deal_list", index=False)
print("Saved", out / "scorecard.xlsx")

## Next ideas to add on the night
- **Organic vs acquired growth:** if segment data or the deep profile splits it out.
- **Leverage check:** net debt ÷ EBITDA trend; is the engine funded by cash or debt?
- **Says vs does:** pull transcripts (`pull.py --transcripts 4`), extract what management promised about M&A, compare with the deal list.
- **Runway:** Deck agent estimate of private targets per vertical ÷ deals per year.